| Step | Component                         | Purpose                                                                                        |
| ---- | --------------------------------- | ---------------------------------------------------------------------------------------------- |
| 1    | Chat Dataset                      | Stores conversation examples for training.                                                     |
| 2    | Train / Validation Split          | Splits the dataset into training and validation sets for model evaluation.                     |
| 3    | Tokenizer                         | Converts text into tokens.                                                                     |
| 4    | `apply_chat_template()`           | Formats conversations into the model's chat style.                                             |
| 5    | Tokenization                      | Converts formatted text into `input_ids` and `attention_mask`.                                 |
| 6    | Tokenized Dataset                 | Final dataset used for training and validation.                                                |
| 7    | `DataCollatorForLanguageModeling` | Dynamically pads sequences and prepares labels for causal language modeling.                   |
| 8    | `BitsAndBytesConfig`              | Configures 4-bit quantization to reduce GPU memory usage.                                      |
| 9    | Load Base Model                   | Loads the pretrained Gemma-3-1B-Instruct model.                                                |
| 10   | `LoraConfig`                      | Defines LoRA hyperparameters (rank, alpha, dropout, target modules).                           |
| 11   | `get_peft_model()`                | Injects LoRA adapters into the frozen base model.                                              |
| 12   | `TrainingArguments`               | Configures epochs, batch size, learning rate, checkpointing, logging, and evaluation strategy. |
| 13   | `Trainer`                         | Handles batching, training, validation, optimization, and checkpoint management.               |
| 14   | Model Training                    | Fine-tunes only the LoRA adapter weights while keeping the base model frozen.                  |
| 15   | Validation (`trainer.evaluate()`) | Evaluates the model on the validation dataset and computes validation loss.                    |
| 16   | Perplexity Calculation            | Computes perplexity from the validation loss to measure language modeling performance.         |
| 17   | `merge_and_unload()` *(Optional)* | Merges LoRA weights into the base model for standalone deployment.                             |
| 18   | `model.push_to_hub()`             | Uploads the fine-tuned (or merged) model to the Hugging Face Hub.                              |
| 19   | `tokenizer.push_to_hub()`         | Uploads the tokenizer and chat template for inference.                                         |
| 20   | Inference                         | Loads the model, applies the chat template, generates responses, and decodes the output.       |


In [ ]:
import kagglehub
path = kagglehub.dataset_download("zsham433/chatdataset")

In [ ]:
import os

for root, dirs, files in os.walk(path):
    for file in files:
        print(os.path.join(root, file))

/root/.cache/kagglehub/datasets/zsham433/chatdataset/versions/1/chatDataset.json


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

# import kagglehub
# zsham433_chatdataset_path = kagglehub.dataset_download('zsham433/chatdataset')

# print('Data source import complete.')


#### bitsandbytes
bitsandbytes is a quantization library used with Hugging Face Transformers. It enables loading and training large language models in 8-bit or 4-bit precision, significantly reducing GPU memory usage. It is commonly used together with LoRA to implement QLoRA.

In [2]:
from huggingface_hub import login
from google.colab import userdata

In [3]:
HF_WRITE_TOKEN = userdata.get("HF_WRITE_TOKEN")
login(token=HF_WRITE_TOKEN)

In [4]:
HF_TOKEN = userdata.get("HF_READ_TOKEN")
login(token=HF_TOKEN)

In [ ]:
# from huggingface_hub import whoami
# print(whoami())

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_READ_TOKEN")

print(HF_TOKEN[:10])

hf_FBmibYr


In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

In [ ]:
# from datasets import load_dataset

# dataset = load_dataset(
#     "json",
#     data_files="/kaggle/input/datasets/zsham433/chatdataset/chatDataset.json"
# )

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files=os.path.join(path, "chatDataset.json")
)

print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 517
    })
})
{'messages': [{'content': 'You are Shafiq AI assistant. You explain concepts clearly, practically, and in a developer-friendly way. Keep responses structured and helpful.', 'role': 'system'}, {'content': 'Explain briefly: What is the AI Mock Interview project in Shafiq?', 'role': 'user'}, {'content': 'The AI Mock Interview project is a full-stack AI application where users enter job details and resume information. The system generates personalized interview questions and provides detailed performance feedback at the end.', 'role': 'assistant'}]}


In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 517
    })
})

In [ ]:
data = dataset['train']
data

Dataset({
    features: ['messages'],
    num_rows: 517
})

In [ ]:
print(dataset["train"].features)

{'messages': List({'content': Value('string'), 'role': Value('string')})}


In [ ]:
print(dataset["train"].column_names)

['messages']


In [ ]:
print(dataset["train"][0])

{'messages': [{'content': 'You are Shafiq AI assistant. You explain concepts clearly, practically, and in a developer-friendly way. Keep responses structured and helpful.', 'role': 'system'}, {'content': 'Explain briefly: What is the AI Mock Interview project in Shafiq?', 'role': 'user'}, {'content': 'The AI Mock Interview project is a full-stack AI application where users enter job details and resume information. The system generates personalized interview questions and provides detailed performance feedback at the end.', 'role': 'assistant'}]}


#### model understand this format
```
`<|System|>`
instruction

`<|User|>`
create a python project

`<|assistant|>`
output form model

In [ ]:
dataset["train"]["messages"][0][0]["content"]

'You are Shafiq AI assistant. You explain concepts clearly, practically, and in a developer-friendly way. Keep responses structured and helpful.'

In [ ]:
dataset["train"]["messages"][0][1]["content"]

'Explain briefly: What is the AI Mock Interview project in Shafiq?'

In [ ]:
dataset["train"]["messages"][0][2]["content"]

'The AI Mock Interview project is a full-stack AI application where users enter job details and resume information. The system generates personalized interview questions and provides detailed performance feedback at the end.'

In [ ]:
dataset["train"]["messages"][0][0]["content"]

'You are Shafiq AI assistant. You explain concepts clearly, practically, and in a developer-friendly way. Keep responses structured and helpful.'

# Data Format   --  apply_chat_template

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")

def format_data(data):

    text = tokenizer.apply_chat_template(
        data["messages"],

        tokenize=False,

        add_generation_prompt=False

           )

    return {"text": text}

dataset = dataset.map(format_data)

# tokenize=False----, here we just formating input not converting inpt into numbers
#  add_generation_prompt=False---- it mean model in training mood so model
# will not generate it will use Answer as well not only question..


In [ ]:
dataset["train"][0]["text"]

'<bos><start_of_turn>user\nYou are Shafiq AI assistant. You explain concepts clearly, practically, and in a developer-friendly way. Keep responses structured and helpful.\n\nExplain briefly: What is the AI Mock Interview project in Shafiq?<end_of_turn>\n<start_of_turn>model\nThe AI Mock Interview project is a full-stack AI application where users enter job details and resume information. The system generates personalized interview questions and provides detailed performance feedback at the end.<end_of_turn>\n'

Gemma ke tokenizer ne system or user ko merge kar diya.

 Gemma ke chat template me system prompt ko alag <start_of_turn>system block me nahi rakha jata. Official Gemma chat template system  or user turn ke andar merge kar deta hai.

In [ ]:
print(dataset.column_names)

# apply_chat_template--- ye krny sy har row ka ik "text" column b ban gia hy ...

{'train': ['messages', 'text']}


In [ ]:
# Train Test SPlit

In [ ]:
dataset = dataset["train"].train_test_split(
    test_size=0.1,
    seed=42
)

# Model Tokenize

In [ ]:
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")

tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(data):
    tokens =  tokenizer(
        data["text"],
        truncation=True,
        max_length=256
    )
    # tokens["labels"] = tokens["input_ids"].copy()

    # teacher forcing ki trha hi ye b model k ander jayer gy shift kr dain gy label ko
    # 1 step or input_ids as input or lable 1 shift k sath as predict  kia jayer ga ..

    return tokens


# def tokenize_fn(data):
#     tokens = tokenizer(
#         data['text'],
#         padding="max_length", #--- ye method sy dynamic padding nhi hoti
#         truncation=True,
#         max_length=256
#     )

dataset = dataset.map(tokenize_fn,batched=True,remove_columns=dataset["train"].column_names)

Map:   0%|          | 0/465 [00:00<?, ? examples/s]

Map:   0%|          | 0/52 [00:00<?, ? examples/s]

`tokenizer.eos_token` (**End Of Sequence token**) ek **special token** hota hai jo model ko batata hai ke **text ya response yahan khatam ho gaya hai**. Fine-tuning ke dauran model ye bhi seekhta hai ke jawab complete hone par EOS token generate karna hai, aur inference ke waqt jab model EOS token generate karta hai to generation stop ho jati hai. Gemma aur Llama jaise decoder-only models me aksar alag `pad_token` nahi hota, isliye batching ke waqt temporary `tokenizer.pad_token = tokenizer.eos_token` set kar dete hain taake padding ho sake, jabke `attention_mask` padding wali positions ko ignore kar deta hai.


In [ ]:
print(dataset["train"].column_names)

['input_ids', 'attention_mask']


In [ ]:
print(tokenizer.eos_token)
print(tokenizer.eos_token_id)

<eos>
1


In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 465
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 52
    })
})

In [ ]:
# from huggingface_hub import whoami
# print(whoami())

# Push Tokenized Data to HF

In [ ]:
dataset.push_to_hub("shafiq433/chatDataset")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 18.9kB / 18.9kB            

...

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 6.64kB / 6.64kB            

README.md:   0%|          | 0.00/407 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/shafiq433/chatDataset/commit/9923181391d78ff7fa9227635e402c62604d7299', commit_message='Upload dataset', commit_description='', oid='9923181391d78ff7fa9227635e402c62604d7299', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/shafiq433/chatDataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='shafiq433/chatDataset'), pr_revision=None, pr_num=None)

# Padding

#### 1) DataCollatorWithPadding
 mostly use with sentimentAnalysis , Spam Detection, Mental Health Classification..

  q k inme lable already hoty hain 0,1,2,3.... or ye label ko nhi disturb krna

In [ ]:
# from transformers import DataCollatorWithPadding

# data_collator = DataCollatorWithPadding(
#     tokenizer=tokenizer,
#     padding=True
# )



#### 2) DataCollatorForLanguageModeling

 labels already maujood hain.

 To naye labels nahi banayega agr already bna liye hain. or padding b kry ga or <pad> ki jga -100 b lgye ga taky model training k wqt ignore krde..

In [ ]:
from transformers import DataCollatorForLanguageModeling

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer = tokenizer,
    mlm=False
)

In [ ]:
# from torch.utils.data import DataLoader

# loader = DataLoader(
#     dataset["train"],
#     batch_size=2,
#     collate_fn=data_collator
# )
# batch = next(iter(loader))

# print(batch.keys())

In [ ]:
# from transformers import default_data_collator

# data_collator = default_data_collator

# Model

`LoRA` freezes the base model in its original precision, while `QLoRA` aggressively compresses the base model into 4-bit precision before applying LoRA adapters.

In [ ]:
!pip uninstall -y bitsandbytes
!pip install -U bitsandbytes

Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (60.7 MB)


In [ ]:
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-1b-it",
    quantization_config = bnb_config, #-------------bnb_config
    device_map={"":0})

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


# Lora Apply

##### LoraConfig

- Ye LoRA ki settings define karta hai.
- Kitna rank (r)
- Alpha kitna hoga
- Dropout kitna hoga
- Kis layers par LoRA lagani hai
- Yani LoRA ka blueprint.
```
```
get_peft_model
- Iska kaam hai:
- Original model ke andar LoRA adapters add karna.
```

Gemma

Layer 1  (Frozen)
   │
   ├── LoRA Adapter (Trainable)

Layer 2 (Frozen)
   │
   ├── LoRA Adapter (Trainable)

Layer 3 (Frozen)
   │
   ├── LoRA Adapter (Trainable)

Layer 4 (Frozen)
   │
   ├── LoRA Adapter (Trainable)

In [ ]:
from peft import LoraConfig,get_peft_model

In [ ]:
lora_config = LoraConfig(  # --- Lora Setting
    r=8, #----------- no Of Trainable Parameters.... comonly use 8,16
    lora_alpha=16, #----------- effect of lora on model
    target_modules=["q_proj","v_proj"], #--- apply Lora on Query,Value
    lora_dropout=0.05, #---- 5% LoRA neurons randomly off.
    bias="none", # ---- wants to apply lora on bias ..(none),(all),(lora_only)
    task_type="CAUSAL_LM" # big models list come in it
)
model = get_peft_model(model,lora_config) # ---  apply on gemma model

nternally PEFT:

- Base model ke parameters freeze kar deta hai.
- LoRA layers add karta hai.
- Sirf LoRA parameters ko requires_grad=True rakhta hai.



```
Transformer Layer

Wq (Frozen)

+
LoRA Adapter (Trainable)

-----------------

Wk (Frozen)

-----------------

Wv (Frozen)

+
LoRA Adapter (Trainable)

-----------------

Wo (Frozen)

### `Now original Gemma will not update only Gemma will update  `

 Query = X × Wq

after Lora

Query = X × (Wq + ΔW) ........... Here ΔW is Lora jo sath mil kr train hojata hy... yani ik chota sas neural system attach hojta hy............ or Wq gemma k weights freeze rhty hain

LoRA kya karta hai?

Us teacher ko replace nahi karta.

Bas uske kaan me ek assistant khada kar deta hai.

# Trainer

In [ ]:
# data = dataset.remove_columns(["messages", "text"])

In [ ]:
from transformers import Trainer, TrainingArguments

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    eval_strategy="epoch",   # ya eval_strategy="epoch"
    eval_steps=None,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    greater_is_better=False,
    output_dir="./gemma_lora", # Model kis folder me save hoga
    per_device_train_batch_size=2, # Ek GPU par batch size
    num_train_epochs=3, # Total kitni baar dataset train hoga
    learning_rate=2e-4, # Learning rate
    logging_steps=10, # Har 10 steps ke baad loss print karo
    save_strategy="epoch", # Har epoch ke baad model save karo
    save_total_limit=2, # Sirf latest 2 checkpoints rakho
    fp16=True, # Mixed Precision (GPU memory kam use hogi)
    gradient_accumulation_steps=1, # Har 1 step k bad optimizer.zero_grad(), backward(), step() yani update hongy
    report_to="none",# Report TensorBoard/WandB ko mat bhejo
    # Random seed
    seed=42
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator, # padding
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'pad_token_id': 1}.


Epoch,Training Loss,Validation Loss
1,1.124564,1.100120
2,0.567080,0.609764
3,0.375300,0.468045


TrainOutput(global_step=699, training_loss=1.160307586789984, metrics={'train_runtime': 260.6147, 'train_samples_per_second': 5.353, 'train_steps_per_second': 2.682, 'total_flos': 503386596801792.0, 'train_loss': 1.160307586789984, 'epoch': 3.0})

In [ ]:
# Evaluation

In [ ]:
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Epoch
0.375300,0.468045,3


{'eval_loss': 0.46804502606391907}


In [ ]:
# Perplexity

## yani kitny words me confuse hota hy ...

In [ ]:
import math

perplexity = math.exp(results["eval_loss"])

print(f"Perplexity: {perplexity:.2f}")

Perplexity: 1.60


```
Sentence
      │
      ▼
Tokenizer
      │
      ▼
Input IDs
      │
      ▼
Model
      │
      ▼
Logits
      │
      ▼
Softmax
      │
      ▼
Predicted Probabilities
      │
      ▼
Compare with Ground Truth (Actual Next Token)
      │
      ▼
Cross Entropy Loss
      │
      ▼
Average over all Tokens
      │
      ▼
Validation Loss
      │
      ▼
Perplexity = e^(Validation Loss)


# Model Merge

In [ ]:
model = trainer.model
model = model.merge_and_unload()

# jo trained LoRA weights hain, unko base model ke weights me permanently merge kar diya jata hai.

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


# Push Model & Tokenizer to HF

In [ ]:
model.push_to_hub("shafiq433/GemmaChatBoat")
tokenizer.push_to_hub("shafiq433/GemmaChatBoat")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...7xeojmn/model.safetensors:   1%|          | 7.48MB /  965MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpcuf1w_bs/tokenizer.json:  48%|####7     | 16.0MB / 33.4MB            

CommitInfo(commit_url='https://huggingface.co/shafiq433/GemmaChatBoat/commit/c5b359f4ac6cea147e6acd4ef152de8ad21539e9', commit_message='Upload tokenizer', commit_description='', oid='c5b359f4ac6cea147e6acd4ef152de8ad21539e9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/shafiq433/GemmaChatBoat', endpoint='https://huggingface.co', repo_type='model', repo_id='shafiq433/GemmaChatBoat'), pr_revision=None, pr_num=None)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("shafiq433/GemmaChatBoat")

print(tokenizer.chat_template is None)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

False


In [ ]:
# from transformers import AutoTokenizer, AutoModelForCausalLM
# import torch

# model_name = "shafiq433/GemmaChatBoat"

# tokenizer = AutoTokenizer.from_pretrained(model_name)

# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     device_map="auto",
#     torch_dtype=torch.float16
# )

# while True:

#     question = input("\nYou: ")

#     if question.lower() in ["exit", "quit"]:
#         break

#     messages = [
#         {"role": "user", "content": question}
#     ]

#     inputs = tokenizer.apply_chat_template(
#         messages,
#         add_generation_prompt=True,
#         tokenize=True,
#         return_dict=True,
#         return_tensors="pt",
#     ).to(model.device)

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=100,
#             do_sample=True,
#             temperature=0.7,
#             top_p=0.9,
#             repetition_penalty=1.1
#         )

#     response = tokenizer.decode(
#         outputs[0][inputs["input_ids"].shape[-1]:],
#         skip_special_tokens=True
#     )

#     print("\nAssistant:", response)

In [6]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
